# Fake News & Source Credibility Detector — Colab (Git-based)

This notebook pulls **all code straight from GitHub** (no Google Drive copy),
so you always run the latest version — no stale-folder traps.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then *Runtime → Run all*.

Repo: https://github.com/Sanjana132/FakeNewsCredibilityAssesor.git


## 1 · Verify the GPU


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.')


## 2 · Clone the repo **fresh** from GitHub

`rm -rf` first so re-running this cell always gives you the latest code
(this is what prevents the stale-code problem). Data/model folders are
recreated by later cells, so nothing important is lost here.


In [ ]:
import os
REPO = 'https://github.com/Sanjana132/FakeNewsCredibilityAssesor.git'
DIR  = '/content/FakeNewsCredibilityAssesor'
!rm -rf $DIR
!git clone --depth 1 $REPO $DIR
os.chdir(DIR)
print('\nNow in:', os.getcwd())
print('Latest commit:')
!git log --oneline -1
# Sanity: confirm the Phase-5 training fix is present
src = open('phase5_deberta.py').read()
print('phase5 fix present:', all(k in src for k in
      ['set_feature_stats', 'init_output_bias', 'Forward-pass sanity check']))


## 3 · Install dependencies + NLTK data


In [ ]:
# requirements.train.txt pulls in requirements.api.txt automatically.
!pip install -q -r requirements.train.txt
import nltk
for pkg in ['stopwords','punkt','punkt_tab','opinion_lexicon']:
    nltk.download(pkg, quiet=True)
print('Dependencies installed.')


## 4 · Phase 1–3 · Build the dataset

Downloads LIAR-2, MultiFC, FEVER, AVeriTeC from HuggingFace and builds
`data/train.csv`, `val.csv`, `test.csv`.

Add `--sample 3000` for a fast smoke run; omit for the full ~90k-row dataset.


In [ ]:
import os
if all(os.path.exists(f'data/{s}.csv') for s in ['train','val','test']):
    print('Data already present — skipping.')
else:
    !python credibility_detector_phases123.py 2>&1 | tail -40


In [ ]:
import pandas as pd
for s in ['train','val','test']:
    df = pd.read_csv(f'data/{s}.csv')
    print(f'{s:5} {len(df):>7,} rows   NaN credibility={df["credibility_score"].isna().sum()}')


## 5 · Phase 4 · TF-IDF baseline (MAE benchmark)


In [ ]:
!python phase4_baseline_1.py --no-shap


## 6 · Phase 5 · Fine-tune DeBERTa-v3 on GPU  ⏳ (main step)

Watch for `✓ Forward-pass sanity check OK on 'cuda' ...` before Epoch 1.
val_MAE should fall each epoch (target ≈ 0.20–0.24 vs baseline ≈ 0.287),
`Pearson_r` should be a real number, and there should be **no `NaN-skip`**.


In [ ]:
!python phase5_deberta.py --train --device cuda


In [ ]:
import json
r = json.load(open('models/deberta_results.json'))
print('Best epoch   :', r['best_epoch'])
print('Best val MAE :', r['best_val_MAE'])
print('Test MAE     :', r['test']['MAE'])
print('Test F1      :', r['test']['Macro_F1'])
print('Baseline MAE :', r['baseline_MAE'])
print('Improvement  :', r['improvement'])
print('Per-dataset  :', json.dumps(r.get('per_dataset', {}), indent=2))


## 7 · Phase 5b · Calibration & uncertainty


In [ ]:
!python phase5b_calibration.py --device cuda


## 8 · Phase 6 · Speaker profiles + context encoder


In [ ]:
!python phase6_speaker_profiler.py
!python context_encoder.py --visualise


## 9 · Phase 7 · Token-level SHAP explanations


In [ ]:
!python phase7_shap_explainer.py --n-examples 12 --n-global 150


## 10 · (Optional) Mistral-7B QLoRA justification model

**Needs an A100 / high-RAM runtime.** Skip on a plain T4.


In [ ]:
# !python llm_finetune.py --train
print('Uncomment above on an A100 runtime.')


## 11 · Save trained artifacts to Google Drive

So the weights survive the runtime disconnecting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
dst = '/content/drive/MyDrive/FakeNews_models'
os.makedirs(dst, exist_ok=True)
for f in ['deberta_best.pt','deberta_results.json','deberta_tokenizer',
          'baseline_tfidf.pkl','baseline_results.json','calibration.json',
          'speaker_profiles.json']:
    src = f'models/{f}'
    if os.path.exists(src):
        d = f'{dst}/{f}'
        if os.path.isdir(src):
            shutil.rmtree(d, ignore_errors=True); shutil.copytree(src, d)
        else:
            shutil.copy(src, d)
        print('saved', f)
print('Done → Drive/MyDrive/FakeNews_models')


## 🔄 Re-pull the latest code (run anytime a fix is pushed)

Updates the code from GitHub **without** deleting your `data/` or `models/`
(those are git-ignored, so `git pull` leaves them untouched). Re-run the
phase cells afterwards.


In [ ]:
import os
os.chdir('/content/FakeNewsCredibilityAssesor')
!git pull --ff-only
!git log --oneline -1


## 12 · Quick inference smoke test


In [ ]:
!python phase5_deberta.py --predict "The unemployment rate fell to a record low." \
    --speaker "Joe Biden" --context "a speech" --device cuda
